In [2]:
from dataclasses import dataclass

@dataclass
class ProteinObj:
    aaSeq: str
    protWeight: float
    associatedGeneID: int

@dataclass
class NaturalGene:
    isoforms: list
    geneID: int
    geneName: str
    organism: str
    DNASequence: str
    spliceAIDonor: list
    spliceAIReceptor: list
    energetics: dict
    chromosome: int

@dataclass
class IsoformGeneBody:
    isoformNumber: str
    associatedProtein: ProteinObj
    fullSequence: str
    codingSeq: str
    relativeAbundance: float
    geneBody: list

@dataclass
class Isoform:
    isoformNumber: str
    associatedProtein: ProteinObj
    fullSequence: str
    codingSeq: str
    relativeAbundance: float


@dataclass
class SyntheticGene:
    AASeq: str
    desc: str
    DNASeqs: list
    degreeOfDegeneracy: int
    vectors: list

@dataclass
class CodonAnalysis:
    transcriptome: str
    AAFreqs: dict
    totalCodons: int
    codonFreqsByLocation: dict
    codonFreqsLit: dict
    usagePerGene: list[dict]
    codonUsageScoreByGene: list[float]

@dataclass
class RareCodonAnalysis:
    organism: str
    transcriptome: str
    totalCodons: int
    codonFreqsByLocation: dict
    codonFreqsLit: dict
    usagePerGene: list[dict]

@dataclass
class GCAnalysis:
    transcriptome: str
    windows: dict
    taggedGC: dict
    taggedGC1: dict
    taggedGC2: dict
    taggedGC3: dict

In [3]:
'''This doc contains all the functions necessary to run the first genetic algorithm'''
import numpy as np
import random
import os, pickle
#from GeneClasses import SyntheticGene, NaturalGene, Isoform, ProteinObj, IsoformGeneBody, codonAnalysis, GCAnalysis
from Standards import totalCodonUsageLit, codonUsageDataForm, defineCodonUsageRef, CPBBaseline, CPBScoreFromCodonSeq
from Standards import gcBaseline, gcBaselineGB, gcReport, codonUsageDataFormGB
from MotifAnalysisLib import generateDownVec, motifAnalysisFragmentMPSharedMemory
#from Standards import idMotifs, defineMotifUsage
from multiprocessing import Pool, shared_memory
from multiprocessing.managers import BaseManager
import scipy.stats

class CustomManager(BaseManager):
    # Nothing to see here
    pass

def motifOfXnt_MP_ShareTranscriptomeReturnDict(x: int, transOfChoice: str):
    motifDict = {}
    # Take a sequence and return a list of all motifs of length x
    if x < 2:
        print("Motif length must be at least 2.")
        return None
    totCombs = 4 ** x
    lst = np.array([[4, 3, 2, 1]], dtype='int8')

    # generate all possible motifs of length x
    i = 1
    while i < x:
        lst = np.append(lst, [generateDownVec(4)], axis=0)
        i += 1
    seqs = np.reshape(np.meshgrid(*lst), (x, -1)).T
    del lst

    print(">>>>>")
    print(seqs)
    print("<<<<<", "total: ", str(totCombs))
    gSet = loadTranscriptome(transOfChoice)

    serializedGenes = []
    for gene in gSet:
        serializedGenes.append(pickle.dumps(gene))
    sharedGSet = shared_memory.ShareableList(serializedGenes[:], name='SHRD_gSet')

    with CustomManager() as manager:
        manager.register('NaturalGene', NaturalGene)
        manager.register('Isoform', Isoform)
        manager.register('ProteinObj', ProteinObj)

        sqList = []
        for s in seqs:
            if len(sqList) < 100:
                sqList.append(s)
            else:
                with Pool(10) as p:
                    #packlist = p.starmap(motifAnalysisFragmentMPSharedMemory, sqList)
                    packlist = p.map(motifAnalysisFragmentMPSharedMemory, sqList)
                for pack in packlist:
                    #returns a tuple [str subseq, dict of occurnces: FIIIIND ME
                    #occVec = {
                    #    'Upstream': [],
                    #    '-50bpUpstream': [],
                    #    '+50bpUpstream': [],
                    #    'Downstream': [],
                    #    '-50bpDownstream': [],
                    #    '+50bpDownstream': [],
                    #    'Exon': [],
                    #    '-50bpExon': [],
                    #    '+50bpExon': [],
                    #    'Intron': [],
                    #    '-50bpIntron': [],
                    #    '+50bpIntron': []
                    #}
                    subStr = pack[0]
                    occVec = pack[1]
                    motifDict[subStr] = occVec
                sqList = []

    # Save the dictionary
    #root = r'C:\Users\Luke\PycharmProjects\GeneRider\MotifsPresent'
    root = r'D:\GeneRiderOutputs\MotifsPresent'
    n = transOfChoice + str(x) + "ntMotifs.pkl"
    path = os.path.join(root, n)
    with open(path, 'wb') as handle:
        pickle.dump(motifDict, handle)

    return motifDict


def chooseTranscriptome():
    # get an input for which of the directories you want as your transcriptome and add it to root
    print("Transcriptomes available: ")
    root = r'C:\Users\Luke\PycharmProjects\GeneRider\Transcriptome'
    for r, d, f in os.walk(root):
        for folder in d:
            print(folder)

    transOfChoice = input("Which transcriptome do you want to use? ")
    return transOfChoice

def seedSols(vec: list, numSols: int):
    #Take a vector and generate a bunch of random solutions
    #Return a list of these solutions
    sols = []
    for i in range(numSols):
        sol = []
        for j in range(len(vec)):
            sol.append(np.random.choice(vec[j]))
        sol = np.array(sol, dtype='int8')
        sols.append(sol)
    return sols

def IOtoFreqVec(ioList: list):
    numSingleRareCodons = 0
    numDoubleRareCodons = 0
    numTripleRareCodons = 0
    numQuadRareCodons = 0
    numQuintRareCodons = 0
    numSextRareCodons = 0
    numSeptRareCodons = 0
    numOctRareCodons = 0
    numNonRareCodons = 0
    numTenRareCodons = 0
    numElevenRareCodons = 0
    numTwelveRareCodons = 0
    window = 12
    windowFrames = 0

    if len(ioList) > window - 1:
        for i in range(window, len(ioList), 1):
            windowFrames += 1
            ioSum = sum(ioList[i - window:i])
            match ioSum:
                case 0:
                    continue
                case 1:
                    numSingleRareCodons += 1
                case 2:
                    numDoubleRareCodons += 1
                case 3:
                    numTripleRareCodons += 1
                case 4:
                    numQuadRareCodons += 1
                case 5:
                    numQuintRareCodons += 1
                case 6:
                    numSextRareCodons += 1
                case 7:
                    numSeptRareCodons += 1
                case 8:
                    numOctRareCodons += 1
                case 9:
                    numNonRareCodons += 1
                case 10:
                    numTenRareCodons += 1
                case 11:
                    numElevenRareCodons += 1
                case 12:
                    numTwelveRareCodons += 1


    rareOccurenceVec = [numSingleRareCodons, numDoubleRareCodons, numTripleRareCodons, numQuadRareCodons,
                        numQuintRareCodons, numSextRareCodons, numSeptRareCodons, numOctRareCodons, numNonRareCodons,
                        numTenRareCodons, numElevenRareCodons, numTwelveRareCodons]
    for i in range(len(rareOccurenceVec)):
        rareOccurenceVec[i] = rareOccurenceVec[i] / windowFrames

    return rareOccurenceVec

def loadAllRefGenes():
    files = []
    root = r'C:\Users\Luke\PycharmProjects\GeneRider\ReferenceGenes'
    for r, d, f in os.walk(root):
        for file in f:
            if '.pkl' in file:
                files.append(os.path.join(r, file))
    genes = []
    for f in files:
        with open(f, 'rb') as infile:
            gene = pickle.load(infile)
            genes.append(gene)
    return genes

def defineRareCodonsRefGenes():
    """Define rare codons using the list of reference genes. Rare Codons are used less than 10% of the time, relative to synonymous codons."""
    totRefCods = 0
    refGenes = loadAllRefGenes()

    codonCounter = {'TTT': 0, 'TCT': 0, 'TAT': 0, 'TGT': 0, 'TTC': 0, 'TCC': 0, 'TAC': 0,
                     'TGC': 0, 'TTA': 0, 'TCA': 0, 'TAA': 0, 'TGA': 0, 'TTG': 0, 'TCG': 0,
                     'TAG': 0, 'TGG': 0, 'CTT': 0, 'CCT': 0, 'CAT': 0, 'CGT': 0, 'CTC': 0,
                     'CCC': 0, 'CAC': 0, 'CGC': 0, 'CTA': 0, 'CCA': 0, 'CAA': 0, 'CGA': 0,
                     'CTG': 0, 'CCG': 0, 'CAG': 0, 'CGG': 0, 'ATT': 0, 'ACT': 0, 'AAT': 0,
                     'AGT': 0, 'ATC': 0, 'ACC': 0, 'AAC': 0, 'AGC': 0, 'ATA': 0, 'ACA': 0,
                     'AAA': 0, 'AGA': 0, 'ATG': 0, 'ACG': 0, 'AAG': 0, 'AGG': 0, 'GTT': 0,
                     'GCT': 0, 'GAT': 0, 'GGT': 0, 'GTC': 0, 'GCC': 0, 'GAC': 0, 'GGC': 0,
                     'GTA': 0, 'GCA': 0, 'GAA': 0, 'GGA': 0, 'GTG': 0, 'GCG': 0, 'GAG': 0,
                     'GGG': 0}

    aaCodonVecs = {
        'S': ['TCT', 'TCC', 'TCA', 'TCG', 'AGT', 'AGC'],
        'L': ['TTA', 'TTG', 'CTT', 'CTC', 'CTA', 'CTG'],
        'C': ['TGT', 'TGC'],
        'W': ['TGG'],
        'E': ['GAA', 'GAG'],
        'D': ['GAT', 'GAC'],
        'P': ['CCT', 'CCC', 'CCA', 'CCG'],
        'V': ['GTT', 'GTC', 'GTA', 'GTG'],
        'N': ['AAT', 'AAC'],
        'M': ['ATG'],
        'K': ['AAA', 'AAG'],
        'Y': ['TAT', 'TAC'],
        'I': ['ATT', 'ATC', 'ATA'],
        'Q': ['CAA', 'CAG'],
        'F': ['TTT', 'TTC'],
        'R': ['CGT', 'CGC', 'CGA', 'CGG', 'AGA', 'AGG'],
        'T': ['ACT', 'ACC', 'ACA', 'ACG'],
        '*': ['TAA', 'TAG', 'TGA'],
        'A': ['GCT', 'GCC', 'GCA', 'GCG'],
        'G': ['GGT', 'GGC', 'GGA', 'GGG'],
        'H': ['CAT', 'CAC']
    }
    aaCounter = {
        'S': 0, 'L': 0, 'C': 0, 'W': 0, 'E': 0, 'D': 0, 'P': 0, 'V': 0, 'N': 0, 'M': 0, 'K': 0,
        'Y': 0, 'I': 0, 'Q': 0, 'F': 0, 'R': 0, 'T': 0, '*': 0, 'A': 0, 'G': 0, 'H': 0
    }
    #RTDDD
    for gene in refGenes:
        for iso in gene.isoforms:
            codingSeqInQ = iso.codingSeq
            maxC = len(codingSeqInQ) // 3
            i = 0
            while i < maxC:
                a = i*3
                codonInQ = codingSeqInQ[a:a+3]
                aaCounter[getAA(codonInQ)] += 1
                totRefCods += 1
                codonCounter[codonInQ] += 1
                i += 1

    rareCodons = []
    for aa in aaCodonVecs.keys():
        vec = aaCodonVecs[aa]
        aaFreq = aaCounter[aa]
        for cod in vec:
            if codonCounter[cod] / aaFreq < 0.1:
                rareCodons.append(cod)

    AAFreqs = {k: v / totRefCods for k, v in aaCounter.items()}
    codonCounter = {k: v / totRefCods for k, v in codonCounter.items()}

    return rareCodons, AAFreqs, codonCounter, totRefCods

def getAA(codon: str):
    #https://gist.github.com/juanfal/09d7fb53bd367742127e17284b9c47bf
    codon = codon.upper()
    codontab = {'TCA': 'S', 'TCC': 'S', 'TCG': 'S', 'TCT': 'S', 'TTC': 'F', 'TTT': 'F', 'TTA': 'L', 'TTG': 'L',
                'TAC': 'Y', 'TAT': 'Y', 'TAA': '*', 'TAG': '*', 'TGC': 'C', 'TGT': 'C', 'TGA': '*', 'TGG': 'W',
                'CTA': 'L', 'CTC': 'L', 'CTG': 'L', 'CTT': 'L', 'CCA': 'P', 'CCC': 'P', 'CCG': 'P', 'CCT': 'P',
                'CAC': 'H', 'CAT': 'H', 'CAA': 'Q', 'CAG': 'Q', 'CGA': 'R', 'CGC': 'R', 'CGG': 'R', 'CGT': 'R',
                'ATA': 'I', 'ATC': 'I', 'ATT': 'I', 'ATG': 'M', 'ACA': 'T', 'ACC': 'T', 'ACG': 'T', 'ACT': 'T',
                'AAC': 'N', 'AAT': 'N', 'AAA': 'K', 'AAG': 'K', 'AGC': 'S', 'AGT': 'S', 'AGA': 'R', 'AGG': 'R',
                'GTA': 'V', 'GTC': 'V', 'GTG': 'V', 'GTT': 'V', 'GCA': 'A', 'GCC': 'A', 'GCG': 'A', 'GCT': 'A',
                'GAC': 'D', 'GAT': 'D', 'GAA': 'E', 'GAG': 'E', 'GGA': 'G', 'GGC': 'G', 'GGG': 'G', 'GGT': 'G'}
    return codontab[codon]

def defineRareCodonsHumanLiterature():
    """This function defines rare codons based on the literature. It returns a list of rare codons, a list of amino acids, a dictionary of codon frequencies, and the total number of codons in the literature dataset."""
    # Defines rare codons defined by <10% usage relative to synonymous codons

    totCods = 80831647

    # Codon frequency per 1000 codons in the genome
    codonFreqsLit = {'TTT': 17.14, 'TCT': 16.93, 'TAT': 12.11, 'TGT': 10.40, 'TTC': 17.48, 'TCC': 17.32, 'TAC': 13.49,
                     'TGC': 10.81, 'TTA': 8.71, 'TCA': 14.14, 'TAA': 0.44, 'TGA': 0.79, 'TTG': 13.44, 'TCG': 4.03,
                     'TAG': 0.35, 'TGG': 11.60, 'CTT': 14.08, 'CCT': 19.31, 'CAT': 11.83, 'CGT': 4.55, 'CTC': 17.81,
                     'CCC': 19.11, 'CAC': 14.65, 'CGC': 8.71, 'CTA': 7.44, 'CCA': 18.92, 'CAA': 14.06, 'CGA': 6.42,
                     'CTG': 36.10, 'CCG': 6.22, 'CAG': 35.53, 'CGG': 10.79, 'ATT': 16.48, 'ACT': 14.26, 'AAT': 18.43,
                     'AGT': 14.05, 'ATC': 18.67, 'ACC': 17.85, 'AAC': 18.30, 'AGC': 19.69, 'ATA': 8.08, 'ACA': 16.52,
                     'AAA': 27.48, 'AGA': 13.28, 'ATG': 21.53, 'ACG': 5.59, 'AAG': 31.77, 'AGG': 12.13, 'GTT': 11.74,
                     'GCT': 18.99, 'GAT': 24.03, 'GGT': 10.83, 'GTC': 13.44, 'GCC': 25.84, 'GAC': 24.27, 'GGC': 19.79,
                     'GTA': 7.66, 'GCA': 17.04, 'GAA': 33.65, 'GGA': 17.12, 'GTG': 25.87, 'GCG': 5.91, 'GAG': 39.67,
                     'GGG': 15.35}

    codonAbsLit = {'TTT': 1385301, 'TCT': 1368632, 'TAT': 978774, 'TGT': 841042, 'TTC': 1413268, 'TCC': 1399962, 'TAC': 1090514,
                   'TGC': 873765, 'TTA': 703680, 'TCA': 1142684, 'TAA': 35218, 'TGA': 63801, 'TTG': 1086777, 'TCG': 325925,
                   'TAG': 28499, 'TGG': 937286, 'CTT': 1138433, 'CCT': 1560898, 'CAT': 956479, 'CGT': 367659, 'CTC': 1439345,
                   'CCC': 1544626, 'CAC': 1184041, 'CGC': 704401, 'CTA': 601662, 'CCA': 1529004, 'CAA': 1136523, 'CGA': 518818,
                   'CTG': 2918400, 'CCG': 503096, 'CAG': 2872161, 'CGG': 871786, 'ATT': 1331901, 'ACT': 1152700, 'AAT': 1489775,
                   'AGT': 1135376, 'ATC': 1508988, 'ACC': 1442511, 'AAC': 1478832, 'AGC': 1591829, 'ATA': 652939, 'ACA': 1335468,
                   'AAA': 2221062, 'AGA': 1073213, 'ATG': 1739992, 'ACG': 452037, 'AAG': 2567940, 'AGG': 980476, 'GTT': 949137,
                   'GCT': 1534685, 'GAT': 1942185, 'GGT': 875715, 'GTC': 1086717, 'GCC': 2088762, 'GAC': 1961667, 'GGC': 1599325,
                   'GTA': 618960, 'GCA': 1377145, 'GAA': 2719693, 'GGA': 1384137, 'GTG': 2090923, 'GCG': 477758, 'GAG': 3206546,
                   'GGG': 1240793}

    aaCodonVecs = {'S': ['TCT', 'TCC', 'TCA', 'TCG', 'AGT', 'AGC'],
                   'L': ['TTA', 'TTG', 'CTT', 'CTC', 'CTA', 'CTG'],
                   'C': ['TGT', 'TGC'],
                   'W': ['TGG'],
                   'E': ['GAA', 'GAG'],
                   'D': ['GAT', 'GAC'],
                   'P': ['CCT', 'CCC', 'CCA', 'CCG'],
                   'V': ['GTT', 'GTC', 'GTA', 'GTG'],
                   'N': ['AAT', 'AAC'],
                   'M': ['ATG'],
                   'K': ['AAA', 'AAG'],
                   'Y': ['TAT', 'TAC'],
                   'I': ['ATT', 'ATC', 'ATA'],
                   'Q': ['CAA', 'CAG'],
                   'F': ['TTT', 'TTC'],
                   'R': ['CGT', 'CGC', 'CGA', 'CGG', 'AGA', 'AGG'],
                   'T': ['ACT', 'ACC', 'ACA', 'ACG'],
                   '*': ['TAA', 'TAG', 'TGA'],
                   'A': ['GCT', 'GCC', 'GCA', 'GCG'],
                   'G': ['GGT', 'GGC', 'GGA', 'GGG'],
                   'H': ['CAT', 'CAC']
                   }

    AAFreqs = {}
    rareCodons = []
    for codon in codonAbsLit:
        aa = getAA(codon)
        if aa in AAFreqs:
            AAFreqs[aa] += codonAbsLit[codon] / totCods
        else:
            AAFreqs[aa] = codonAbsLit[codon] / totCods

    for aa in AAFreqs:
        codons = aaCodonVecs[aa]
        for i in range(len(codons)):
            codon = codons[i]
            codFreq = codonAbsLit[codon] / totCods
            if codFreq / AAFreqs[aa] < 0.1:
                rareCodons.append(codon)

    return rareCodons, AAFreqs, codonAbsLit, totCods

def defineRareCodons():
    basePath = r'C:\Users\Luke\PycharmProjects\GeneRider\RefData'
    lName = 'litRareCodons.pkl'
    rName = 'refRareCodons.pkl'
    lPath = os.path.join(basePath, lName)
    rPath = os.path.join(basePath, rName)


    litPack = defineRareCodonsHumanLiterature()
    """
    litRareCodons = litPack[0]
    litAAFreqs = litPack[1]
    litCodonCounter = litPack[2]
    litTotCods = litPack[3]
    """

    refPack = defineRareCodonsRefGenes()
    """
    refRareCodons = refPack[0]
    refAAFreqs = refPack[1]
    refCodonCounter = refPack[2]
    refTotCods = refPack[3]
    """

    with open(lPath, 'wb') as infile:
        pickle.dump(litPack, infile)
    infile.close()
    with open(rPath, 'wb') as infile:
        pickle.dump(refPack, infile)
    infile.close()

    if len(litPack[0]) != len(refPack[0]):
        print('Warning! The number of rare codons in the literature does not match the number of rare codons in the reference genes.')
        print("Rare codons form literature: ", litPack[0])
        print("Rare codons from reference genes: ", refPack[0])

    return refPack[0]

def rareCodonDataFormRef():
    #Analyze the reference genes and determine the frequency and clustering of rare codons
    refGenes = loadAllRefGenes()
    rareCodons = defineRareCodonsHumanLiterature()
    overallCodonCount = 0
    overallRareCodonCount = 0
    numSingleRareCodons = 0
    numDoubleRareCodons = 0
    numTripleRareCodons = 0
    numQuadRareCodons = 0
    numQuintRareCodons = 0
    numSextRareCodons = 0
    numSeptRareCodons = 0
    numOctRareCodons = 0
    numNonRareCodons = 0
    numTenRareCodons = 0
    numElevenRareCodons = 0
    numTwelveRareCodons = 0
    window = 36
    windowFrames = 0
    freqVec = []

    for gene in refGenes:
        for iso in gene.isoforms:
            ioList = []
            codList = codingSeqToCodonSeq(iso.codingSeq)
            for codon in codList:
                overallCodonCount += 1
                if codon in rareCodons:
                    overallRareCodonCount += 1
                    ioList.append(1)
                else:
                    ioList.append(0)
            freqVec.append(sum(ioList) / len(ioList))
            if len(ioList) >= window:
                for i in range(window, len(codList), 1):
                    windowFrames += 1
                    ioSum = sum(ioList[i-window:i])
                    match ioSum:
                        case 0:
                            continue
                        case 1:
                            numSingleRareCodons += 1
                        case 2:
                            numDoubleRareCodons += 1
                        case 3:
                            numTripleRareCodons += 1
                        case 4:
                            numQuadRareCodons += 1
                        case 5:
                            numQuintRareCodons += 1
                        case 6:
                            numSextRareCodons += 1
                        case 7:
                            numSeptRareCodons += 1
                        case 8:
                            numOctRareCodons += 1
                        case 9:
                            numNonRareCodons += 1
                        case 10:
                            numTenRareCodons += 1
                        case 11:
                            numElevenRareCodons += 1
                        case 12:
                            numTwelveRareCodons += 1
        rareOccurenceVec = [numSingleRareCodons, numDoubleRareCodons, numTripleRareCodons, numQuadRareCodons, numQuintRareCodons, numSextRareCodons, numSeptRareCodons, numOctRareCodons, numNonRareCodons, numTenRareCodons, numElevenRareCodons, numTwelveRareCodons]
        for i in range(len(rareOccurenceVec)):
            rareOccurenceVec[i] = rareOccurenceVec[i] / windowFrames
        return overallRareCodonCount / overallCodonCount, freqVec, rareOccurenceVec

def solVecToCodonSeq(solVec: np.array, AASeq: str):
    #Take a solution vector and an AA sequence and return a DNA sequence
    #The solution vector is a list of integers which correspond to the index of the codon in the degenVec
    #The AA sequence is a string of amino acids
    #The DNA sequence is a string of nucleotides
    codons = []
    for i in range(len(solVec)):
        codons.append(getCodonVecs(AASeq[i]))
    return codons

def codingSeqToCodonSeq(codingSeq: str):
    #Take a coding sequence and return a list of codons
    codons = []
    for i in range(len(codingSeq)//3):
        codons.append(codingSeq[i*3:i*3+3])
    return codons

def getCodonVecs(AA: str):
    AA = AA.upper()
    codonTable = {'S': ['TCT', 'TCC', 'TCA', 'TCG', 'AGT', 'AGC'],
                  'L': ['TTA', 'TTG', 'CTT', 'CTC', 'CTA', 'CTG'],
                  'C': ['TGT', 'TGC'],
                  'W': ['TGG'],
                  'E': ['GAA', 'GAG'],
                  'D': ['GAT', 'GAC'],
                  'P': ['CCT', 'CCC', 'CCA', 'CCG'],
                  'V': ['GTT', 'GTC', 'GTA', 'GTG'],
                  'N': ['AAT', 'AAC'],
                  'M': ['ATG'],
                  'K': ['AAA', 'AAG'],
                  'Y': ['TAT', 'TAC'],
                  'I': ['ATT', 'ATC', 'ATA'],
                  'Q': ['CAA', 'CAG'],
                  'F': ['TTT', 'TTC'],
                  'R': ['CGT', 'CGC', 'CGA', 'CGG', 'AGA', 'AGG'],
                  'T': ['ACT', 'ACC', 'ACA', 'ACG'],
                  '*': ['TAA', 'TAG', 'TGA'],
                  'A': ['GCT', 'GCC', 'GCA', 'GCG'],
                  'G': ['GGT', 'GGC', 'GGA', 'GGG'],
                  'H': ['CAT', 'CAC']}
    return codonTable[AA]

def growthAndMutation(vec, baseVec, numChildren: int, mutationChance: float = 0.10):
    nextGen = []
    for i in range(numChildren):
        child = []
        for j in range(len(vec)):
            if random.random() < mutationChance:
                vec[j] = random.randrange(0, max(baseVec[j]), 1)
            else:
                child.append(vec[j])
        nextGen.append(child)
    return nextGen

def saveGen(gen: list, num: int):
    import pickle
    import os
    import datetime

    todaysDate = datetime.date.today()
    name = str(todaysDate) + 'gen' + str(num) + r'.pkl'
    root = r'C:\Users\Luke\PycharmProjects\GeneRider\GenAlgOut'
    fname = os.path.join(root, name)
    with open(fname, 'wb') as infile:
        pickle.dump(gen, infile)
    infile.close()

def loadMotifScoringMatrix(transOfChoice: str, lengthOfMotif: int):
    import pickle
    import os
    root = r'C:\Users\Luke\PycharmProjects\GeneRider\MotifAnalyses'
    f = transOfChoice + str(lengthOfMotif) + 'ntMotifs.pkl'
    filename = os.path.join(root, f)
    with open(filename, 'rb') as infile:
        scoringMatrix = pickle.load(infile)
    infile.close()
    return scoringMatrix

def checkForMotifDicts(transOfChoice: str):
    import os
    root = r'C:\Users\Luke\PycharmProjects\GeneRider\MotifAnalyses'
    files = []
    corrFiles = []
    ntCounts = []
    for r, d, f in os.walk(root):
        for file in f:
            if '.pkl' in file:
                files.append(os.path.join(r, file))
    for f in files:
        if transOfChoice in f:
            corrFiles.append(f)
    for cFile in corrFiles:
        endInd = cFile.find('ntMotifScoringMatrix.pkl') - 11
        startInd = cFile.find(transOfChoice) + len(transOfChoice) - 1
        substr = cFile[startInd:endInd]
        numStr = ''
        for char in substr:
            if char.isdigit():
                numStr = numStr + char
        ntCounts.append(int(numStr))
    return ntCounts


def checkForMotifCharts(transOfChoice: str, x: int):
    import os
    #FEND ME
    #root = r'C:\Users\Luke\PycharmProjects\GeneRider\MotifsPresent'
    root = r'D:\GeneRiderOutputs\MotifsPresent'
    files = []
    corrFiles = []
    for r, d, f in os.walk(root):
        for file in f:
            if '.pkl' in file:
                files.append(os.path.join(r, file))
    for f in files:
        if transOfChoice in f:
            corrFiles.append(f)
    for cFile in corrFiles:
        if str(x) in cFile:
            return True
        else:
            return False


def statsFromMotifDicts(d: dict):
    from numpy import percentile
    # Dict Values are a ______________
    # occVec = {
    #    'Upstream': [],
    #    '-50bpUpstream': [],
    #    '+50bpUpstream': [],
    #    'Downstream': [],
    #    '-50bpDownstream': [],
    #    '+50bpDownstream': [],
    #    'Exon': [],
    #    '-50bpExon': [],
    #    '+50bpExon': [],
    #    'Intron': [],
    #    '-50bpIntron': [],
    #    '+50bpIntron': []
    # }
    # Dict values are a tuple of [occurences in genome, occurences in transcriptome/ # of transcripts]
    onePack = []
    tupac = []
    for v in d.values():
        onePack.append(v[0])
        tupac.append(v[1])
    #5 number summary
    oPacQuarts = percentile(onePack, [25, 50, 75])
    tPacQuarts = percentile(tupac, [25, 50, 75])
    oPacMin = min(onePack)
    oPacMax = max(onePack)
    tPacMin = min(tupac)
    tPacMax = max(tupac)
    retOne = [oPacMin, oPacQuarts[0], oPacQuarts[1], oPacQuarts[2], oPacMax]
    retTwo = [tPacMin, tPacQuarts[0], tPacQuarts[1], tPacQuarts[2], tPacMax]
    return [retOne, retTwo]

def loadTranscriptome(transcriptome: str):
    import os
    import pickle
    files = []
    root = os.path.join(r'C:\Users\Luke\PycharmProjects\GeneRider\Transcriptome' , transcriptome)
    print(root)

    for r, d, f in os.walk(root):
        for file in f:
            if '.pkl' in file:
                files.append(os.path.join(r, file))
    genes = []
    for f in files:
        with open(f, 'rb') as infile:
            gene = pickle.load(infile)
            genes.append(gene)
    return genes

def generateMotifChart(transOfChoice: str, lengthOfMotif: int):
    import os
    import pickle
    genes = loadTranscriptome(transOfChoice)
    motifs = {}
    for gene in genes:
        a = 0
        while a < len(gene.DNASequence) - lengthOfMotif:
            motif = gene.DNASequence[a:a+lengthOfMotif]
            if motif not in motifs.keys():
                motifs[motif] = [1, 0]
            else:
                motifs[motif][0] += 1
            a += 1
        for iso in gene.isoforms:
            i = 0
            while i < len(iso.codingSeq) - lengthOfMotif:
                motif = iso.codingSeq[i:i+lengthOfMotif]
                if motif not in motifs.keys():
                    motifs[motif] = [0, 1]
                else:
                    motifs[motif][1] += 1
                i += 1
                #FUND ME
    #root = r'C:\Users\Luke\PycharmProjects\GeneRider\MotifsPresent'
    root = r'D:\GeneRiderOutputs\MotifsPresent'
    n = transOfChoice + str(lengthOfMotif) + "ntMotifs.pkl"
    path = os.path.join(root, n)
    with open(path, 'wb') as handle:
        pickle.dump(motifs, handle)
    return motifs

def chartAgainstReferenceGenes(chart: dict, length: int):
    import statistics
    genes = loadAllRefGenes()
    scoreVec = []
    for gene in genes:
        for iso in gene.isoforms:
            codingSequence = iso.codingSeq
            for i in range(len(codingSequence) - length):
                subSeq = codingSequence[i:i+length]
                score = 0
                for j in range(len(chart)):
                    if subSeq in chart:
                        b = chart[subSeq]
                        score += b[1]
                scoreVec.append(score)
    return min(scoreVec), statistics.mean(scoreVec), max(scoreVec), statistics.stdev(scoreVec)

def loadMotifChart(transOfChoice: str, x: int):
    import pickle
    import os
    #FOND ME
    #root = r'C:\Users\Luke\PycharmProjects\GeneRider\MotifsPresent'
    root = r'D:\GeneRiderOutputs\MotifsPresent'
    n = transOfChoice + str(x) + "ntMotifs.pkl"
    path = os.path.join(root, n)
    with open(path, 'rb') as infile:
        motifs = pickle.load(infile)
    infile.close()
    return motifs


def genAlg(oriGene: SyntheticGene, *, numSols: int = 100, numGens: int = 100):
    #Take the degenVecs and randomly choose 100 to 1000 specific solutions using rand()
    #Then, score the solutions based on codon rareness, epigenetic motifs, splicing, etc.
    #pass the top 25% to the next generation and introduce copies with a new synonymous codon in a random position.
    #Bonus points if you can traceback to deleterious mutations
    # Todo - set criteria for this genetic algorithm or at least have slots for it
    print("Choose the transcriptome you want to score synthetic sequences against.")
    transOfChoice = chooseTranscriptome()

    BaseVector = oriGene.vectors
    codonKey = oriGene.AASeq
    #Create a bunch of seed solutions
    g0 = seedSols(BaseVector, numSols)

    #Send them to generation 1
    """Gen 1 -- Rare Codons"""
    g1 = []
    #get the RareCodonsFromTheLiterature
    # TODO - make defineRareCodons map to the D drive
    listRareCodons = defineRareCodons()
    rareCodonDataForm = rareCodonDataFormRef()
    for sol in g0:
        # From the outputs of gen 1, if there are outputs, generate the starters of gen 2
        if gen1(solVecToCodonSeq(sol, codonKey), listRareCodons, rareCodonDataForm):
            packet = growthAndMutation(sol, BaseVector, 10)
            for element in packet:
                g1.append(element)
    del listRareCodons
    del rareCodonDataForm
    saveGen(g1, 1)

    # occVec = {
    #    'Upstream': [],
    #    '-50bpUpstream': [],
    #    '+50bpUpstream': [],
    #    'Downstream': [],
    #    '-50bpDownstream': [],
    #    '+50bpDownstream': [],
    #    'Exon': [],
    #    '-50bpExon': [],
    #    '+50bpExon': [],
    #    'Intron': [],
    #    '-50bpIntron': [],
    #    '+50bpIntron': []
    #    'Splice'
    # }

    """Gen 2 -- Total Codon Usage"""
    g2 = []
    #Get a codonUsageScore baseline
    codAna = codonUsageDataFormGB(transOfChoice)
    #codonUseDataForm = codonUsageDataFormRef()
    #codonUsageFreqFromRef, codonUsageFreqFromLit = defineCodonUsageRef()
    #codonUsageTrendFromRefGenes = codonUseDataForm[2]
    for sol in g1:
        # From the outputs of gen 2, if there are outputs, generate the starters of gen 3
        if gen2(solVecToCodonSeq(sol, codonKey), codAna):
            # Send to generation 2
            g2.append(growthAndMutation(sol, BaseVector, 10))

    del codAna
    saveGen(g2, 2)
    del g1

    #Update to work with gene bodies
    """Gen 3 -- CPB"""
    g3 = []
    #Get a CPB score baseline
    cpbFreqVec = CPBBaseline()

    for sol in g2:
        # From the outputs of gen 3, if there are outputs, generate the starters of gen 4
        if gen3(solVecToCodonSeq(sol, codonKey), min(cpbFreqVec)):
            # Send to generation 3
            g3.append(growthAndMutation(sol, BaseVector, 10))

    del cpbFreqVec
    saveGen(g3, 3)
    del g2


    """Gen 4 -- GC content"""
    g4 = []
    #Get a GC content baseline
    baseline = gcBaselineGB(transOfChoice)
    for sol in g3:
        #return [gcContent, gcContent1, gcContent2, gcContent3, localGCContentVec, localGC1ContentVec, localGC2ContentVec, localGC3ContentVec]

        # From the outputs of gen 4, if there are outputs, generate the starters of gen 5
        if gen4(solVecToCodonSeq(sol, codonKey), baseline):
            # Send to generation 4
            g4.append(growthAndMutation(sol, BaseVector, 10))

    del baseline
    saveGen(g4, 4)
    del g3

    #"""Gen 4 -- GC content"""
    #g4 = []
    #Get a GC content baseline
    #baseline = gcBaseline()
    #for sol in g3:
    #    #return [gcContent, gcContent1, gcContent2, gcContent3, localGCContentVec, localGC1ContentVec, localGC2ContentVec, localGC3ContentVec]
    #
    #    # From the outputs of gen 4, if there are outputs, generate the starters of gen 5
    #    if gen4(solVecToCodonSeq(sol, codonKey), baseline):
    #        # Send to generation 4
    #        g4.append(growthAndMutation(sol, BaseVector, 10))
    #
    #del baseline
    #saveGen(g4, 4)
    #del g3

    """Gen 5 -- Over and Underrepresented Motifs By Full Calculation"""
    g5 = []

    #Get a scoring matrix for motifs. Base it on the exons of the reference genes
    vec = checkForMotifDicts(transOfChoice)
    for d in vec:
        #load the scoring matrix for that length
        scoringMatrix = loadMotifScoringMatrix(transOfChoice, d)
        statPack = statsFromMotifDicts(scoringMatrix)
        #########FIX ALL PARTS OF THE PROGRAM THAT INTERFACE WITH MOTIF DICTIONARIES
        for sol in g4:
            if gen5(solVecToCodonSeq(sol, codonKey), scoringMatrix, d, statPack):
                g5.append(growthAndMutation(sol, BaseVector, 10))

    del scoringMatrix
    del statPack
    saveGen(g5, 5)
    del g4

    """Gen 6 --- Repeated motifs by inference"""
    g6 = []
    scoreVec = {}
    for x in range(10, 20):
        chart = None
        if not checkForMotifCharts(transOfChoice, x):
            chart = motifOfXnt_MP_ShareTranscriptomeReturnDict(x, transOfChoice)
            #chart = generateMotifChart(transOfChoice, x)
            #return min(scoreVec), statistics.mean(scoreVec), max(scoreVec), statistics.stdev(scoreVec)
            pack = chartAgainstReferenceGenes(chart, x)
            pck = [pack[0], pack[1], pack[2], pack[3]]
            scoreVec[x] = pck
        else:
            chart = loadMotifChart(transOfChoice, x)
            pack = chartAgainstReferenceGenes(chart, x)
            pck = [pack[0], pack[1], pack[2], pack[3]]
            scoreVec[x] = pck

    minVec = {}
    meanVec = {}
    maxVec = {}
    for i in range(len(scoreVec)):
        leng = scoreVec[i][0]
        inter = scoreVec[i][1]
        minVec[leng] = inter[0]
        meanVec[leng] = inter[1]
        maxVec[leng] = inter[2]
    #mini
    mini = max(minVec, key=minVec.get)
    #maxi
    maxi = min(maxVec, key=maxVec.get)

    for sol in g5:
        for x in range(10, 20):
            chart = None
            if not checkForMotifCharts(transOfChoice, x):
                chart = motifOfXnt_MP_ShareTranscriptomeReturnDict(x, transOfChoice)
                #chart = generateMotifChart(transOfChoice, x)
            else:
                chart = loadMotifChart(transOfChoice, x)
            if gen6(solVecToCodonSeq(sol, codonKey), chart, x, mini, maxi):
                g6.append(growthAndMutation(sol, BaseVector, 10))
    del chart
    saveGen(g6, 6)
    del g5

    """CollapsePopulation"""
    g6 = collapsePopulation(g6)
    g6 = filterCollapsedThroughPreviousGens(g6, codonKey= codonKey, BaseVector = BaseVector, transOfChoice=transOfChoice)

    """Save data"""
    saveRun(g6, codonKey)

def saveRun(survivors: list, codonKey: str):
    import pickle
    import os
    root = r'C:\Users\Luke\PycharmProjects\GeneRider\GenAlgRuns'
    files = []
    corrFiles = []
    nums = []
    for r, d, f in os.walk(root):
        for file in f:
            if '.pkl' in file:
                files.append(os.path.join(r, file))
    for file in files:
        #Strip extension
        f = os.path.splitext(file)[0]
        #Remove numbers
        seq = ''.join([i for i in f if not i.isdigit()])
        if seq == codonKey:
            corrFiles.append(file)
    for corFile in corrFiles:
        nums.append(int(''.join([i for i in corFile if i.isdigit()])))
    n = 0
    for num in nums:
        if num > n:
            n = num
    n += 1
    name = codonKey + str(n) + r'.pkl'
    root = r'C:\Users\Luke\PycharmProjects\GeneRider\GenAlgRuns'
    fname = os.path.join(root, name)
    with open(fname, 'wb') as infile:
        pickle.dump(survivors, infile)
    infile.close()

def mean_confidence_interval(data, confidence=0.95):
    a = 1.0 * np.array(data)
    n = len(a)
    m, se = np.mean(a), scipy.stats.sem(a)
    h = se * scipy.stats.t.ppf((1 + confidence) / 2., n - 1)
    return m, m - h, m + h

def gen1(codonSeq: list, listRareCods: list, dataForm: tuple):
    #Translate vec to nucs
    tf = True
    avgRareCodonFreq = dataForm[0]
    freqVec = dataForm[1]
    rareOccurenceVec = dataForm[2]
    maxInARow = 0
    for i in range(len(rareOccurenceVec)):
        if rareOccurenceVec[i] == 0:
            maxInARow = i
            break

    lesser = min(avgRareCodonFreq, min(freqVec))

    numRareCodons = 0
    rareIO = []
    for codon in codonSeq:
        if codon in listRareCods:
            numRareCodons += 1
            rareIO.append(1)
        else:
            rareIO.append(0)
    rareOccurenceVecInQ = IOtoFreqVec(rareIO)

    inARow = 0
    for i in range(len(rareOccurenceVecInQ)):
        if rareOccurenceVecInQ[i] == 0:
            inARow = i
            break

    if inARow > maxInARow:
        tf = False

    if numRareCodons / len(codonSeq) > lesser:
        tf = False
    return tf

def gen2(codonSeq: list, codonAnalysisObj: codonAnalysis):
    tf = False
    m, l, h = mean_confidence_interval(codonAnalysisObj.codonUsageScoreByGene)
    predUse = totalCodonUsageLit(codonSeq)
    if l < predUse < h:
        tf = True
    return tf

def gen3(codonSeq: list, cpbThresh: float):
    tf = True
    if CPBScoreFromCodonSeq(codonSeq) < cpbThresh:
        tf = False
    return tf

def gen4(codonSeq: list, gcData: GCAnalysis):
    tf = True
    #Get a string of the codon sequence
    #TODO - restructure this so the sequence is tagged with probable gene body structures
    stringSeq = ''
    for element in codonSeq:
        stringSeq = stringSeq + element
    gcPack = gcReport(stringSeq)
    gcContent = gcPack[0]
    gcContent1 = gcPack[1]
    gcContent2 = gcPack[2]
    gcContent3 = gcPack[3]
    localGCContentVec = gcPack[4]
    localGC1ContentVec = gcPack[5]
    localGC2ContentVec = gcPack[6]
    localGC3ContentVec = gcPack[7]
    #return gcs, gc1s, gc2s, gc3s, maxLocalGCs, minLocalGCs, maxLocalGC1s, minLocalGC1s, maxLocalGC2s, minLocalGC2s, maxLocalGC3s, minLocalGC3s

    if gcContent < min(gcData[0]):
        tf = False
    if gcContent > max(gcData[0]):
        tf = False
    if gcContent1 < min(gcData[1]):
        tf = False
    if gcContent1 > max(gcData[1]):
        tf = False
    if gcContent2 < min(gcData[2]):
        tf = False
    if gcContent2 > max(gcData[2]):
        tf = False
    if gcContent3 < min(gcData[3]):
        tf = False
    if gcContent3 > max(gcData[3]):
        tf = False
    if max(localGCContentVec) > max(gcData[4]):
        tf = False
    if min(localGCContentVec) < min(gcData[5]):
        tf = False
    if max(localGC1ContentVec) > max(gcData[6]):
        tf = False
    if min(localGC1ContentVec) < min(gcData[7]):
        tf = False
    if max(localGC2ContentVec) > max(gcData[8]):
        tf = False
    if min(localGC2ContentVec) < min(gcData[9]):
        tf = False
    if max(localGC3ContentVec) > max(gcData[10]):
        tf = False
    if min(localGC3ContentVec) < min(gcData[11]):
        tf = False
    return tf


"""
def gen4(codonSeq: list, gcData: tuple):
    tf = True
    stringSeq = ''
    for element in codonSeq:
        stringSeq = stringSeq + element
    gcPack = gcReport(stringSeq)
    gcContent = gcPack[0]
    gcContent1 = gcPack[1]
    gcContent2 = gcPack[2]
    gcContent3 = gcPack[3]
    localGCContentVec = gcPack[4]
    localGC1ContentVec = gcPack[5]
    localGC2ContentVec = gcPack[6]
    localGC3ContentVec = gcPack[7]
    #return gcs, gc1s, gc2s, gc3s, maxLocalGCs, minLocalGCs, maxLocalGC1s, minLocalGC1s, maxLocalGC2s, minLocalGC2s, maxLocalGC3s, minLocalGC3s

    if gcContent < min(gcData[0]):
        tf = False
    if gcContent > max(gcData[0]):
        tf = False
    if gcContent1 < min(gcData[1]):
        tf = False
    if gcContent1 > max(gcData[1]):
        tf = False
    if gcContent2 < min(gcData[2]):
        tf = False
    if gcContent2 > max(gcData[2]):
        tf = False
    if gcContent3 < min(gcData[3]):
        tf = False
    if gcContent3 > max(gcData[3]):
        tf = False
    if max(localGCContentVec) > max(gcData[4]):
        tf = False
    if min(localGCContentVec) < min(gcData[5]):
        tf = False
    if max(localGC1ContentVec) > max(gcData[6]):
        tf = False
    if min(localGC1ContentVec) < min(gcData[7]):
        tf = False
    if max(localGC2ContentVec) > max(gcData[8]):
        tf = False
    if min(localGC2ContentVec) < min(gcData[9]):
        tf = False
    if max(localGC3ContentVec) > max(gcData[10]):
        tf = False
    if min(localGC3ContentVec) < min(gcData[11]):
        tf = False
    return tf
"""

def gen5(codonSeq: list, scoringMatrix: dict, lengthOfMotif: int, statPack: list):
    #
    tf = False
    stringSeq = ''
    score = 0.0
    for element in codonSeq:
        stringSeq = stringSeq + element

    #motifs = idMotifs(stringSeq)
    for i in range(len(stringSeq) - lengthOfMotif):
        motif = stringSeq[i:i+lengthOfMotif]

        if motif not in scoringMatrix.keys():
            raise Exception("Motif not found in scoring matrix")

        if motif in scoringMatrix.keys():
            sPack = scoringMatrix[motif]

            score += sPack[1]
    #retOne = [oPacMin, oPacQuarts[0], oPacQuarts[1], oPacQuarts[2], oPacMax]
    #retTwo = [tPacMin, tPacQuarts[0], tPacQuarts[1], tPacQuarts[2], tPacMax]
    if score < statPack[1][4] and score > statPack[1][0]:
        tf = True
    return tf

def gen6(codonSeq: list, chart: dict, lengthOfMotif: int, minimum: float, maximum: float):
    #Take a codonSeq and a chart of motifs and return a boolean
    #If the codonSeq has a motif that is present in the chart, return True
    #Else, return False
    tf = False
    stringSeq = ''
    score = 0.0
    for element in codonSeq:
        stringSeq = stringSeq + element
    #motifs = idMotifs(stringSeq)
    motifsInSeq = []
    for i in range(len(stringSeq) - lengthOfMotif):
        motif = stringSeq[i:i+lengthOfMotif]
        motifsInSeq.append(motif)

    for motif in motifsInSeq:
        if motif in chart.keys():
            score += chart[motif][1]

    if score < maximum and score > minimum:
        tf = True
    return tf

def collapsePopulation(individuals: list, BaseVec):
    weighted_indivs = {}
    for ind in individuals:
        if ind in weighted_indivs:
            weighted_indivs[ind] += 1
        else:
            weighted_indivs[ind] = 1
    extras = []
    for weight, ind in weighted_indivs.items():
        numchild = weight * 10
        children = growthAndMutation(ind, BaseVec, numchildren = numchild, mutationChance=0.20)
        for child in children:
            if child not in individuals:
                extras.append(child)
    newPop = individuals + extras
    return newPop

"""
    for sol in g7:
        if gen8(solVecToCodonSeq(sol, codonKey)):
            g8.append(growthAndMutation(sol, BaseVector, 10))
"""

def filterCollapsedThroughPreviousGens(individuals:list, codonKey, BaseVector, transOfChoice):
    """Check to make sure you're still in bounds and prune failures"""
    listRareCodons = defineRareCodons()
    rareCodonDataForm = rareCodonDataFormRef()
    p1 = []
    for ind in individuals:
        if gen1(solVecToCodonSeq(ind, codonKey), listRareCodons, rareCodonDataForm):
            p1.append(ind)
    del listRareCodons
    del rareCodonDataForm

    p2 = []
    # Get a codonUsageScore baseline
    codAna = codonUsageDataFormGB(transOfChoice)
    for sol in p1:
        # From the outputs of gen 2, if there are outputs, generate the starters of gen 3
        if gen2(solVecToCodonSeq(sol, codonKey), codAna):
            # Send to generation 2
            p2.append(growthAndMutation(sol, BaseVector, 10))

    del codAna
    del p1

    p3 = []
    cpbFreqVec = CPBBaseline()
    for sol in p2:
        if gen3(solVecToCodonSeq(sol, codonKey), min(cpbFreqVec)):
            p3.append(growthAndMutation(sol, BaseVector, 10))
    del cpbFreqVec
    del p2

    p4 = []
    baseline = gcBaseline()
    for sol in p3:
        if gen4(solVecToCodonSeq(sol, codonKey), baseline):
            p4.append(growthAndMutation(sol, BaseVector, 10))

    del baseline
    del p3

    p5 = []
    vec = checkForMotifDicts(transOfChoice)
    for d in vec:
        # load the scoring matrix for that length
        scoringMatrix = loadMotifScoringMatrix(transOfChoice, d)
        statPack = statsFromMotifDicts(scoringMatrix)
        for sol in p4:
            if gen5(solVecToCodonSeq(sol, codonKey), scoringMatrix, d, statPack):
                p5.append(growthAndMutation(sol, BaseVector, 10))
    del scoringMatrix
    del statPack
    del p4

    p6 = []
    scoreVec = {}
    for x in range(10, 20):
        chart = None
        if not checkForMotifCharts(transOfChoice, x):
            #chart = generateMotifChart(transOfChoice, x)
            chart = motifOfXnt_MP_ShareTranscriptomeReturnDict(x, transOfChoice)
            # return min(scoreVec), statistics.mean(scoreVec), max(scoreVec), statistics.stdev(scoreVec)
            pack = chartAgainstReferenceGenes(chart, x)
            pck = [pack[0], pack[1], pack[2], pack[3]]
            scoreVec[x] = pck
        else:
            chart = loadMotifChart(transOfChoice, x)
            pack = chartAgainstReferenceGenes(chart, x)
            pck = [pack[0], pack[1], pack[2], pack[3]]
            scoreVec[x] = pck

    minVec = {}
    meanVec = {}
    maxVec = {}
    for i in range(len(scoreVec)):
        leng = scoreVec[i][0]
        inter = scoreVec[i][1]
        minVec[leng] = inter[0]
        meanVec[leng] = inter[1]
        maxVec[leng] = inter[2]
    # mini
    mini = max(minVec, key=minVec.get)
    # maxi
    maxi = min(maxVec, key=maxVec.get)
    """
            if not checkForMotifCharts(transOfChoice, x):
            chart = generateMotifChart(transOfChoice, x)
    """
    for sol in p5:
        for x in range(10, 20):
            chart = None
            if not checkForMotifCharts(transOfChoice, x):
                chart = motifOfXnt_MP_ShareTranscriptomeReturnDict(x, transOfChoice)
                #chart = generateMotifChart(transOfChoice, x)
            else:
                chart = loadMotifChart(transOfChoice, x)
            if gen6(solVecToCodonSeq(sol, codonKey), chart, x, mini, maxi):
                p6.append(growthAndMutation(sol, BaseVector, 10))
    del chart
    del p5



    return p6

def chartAgainstReferenceGenes(chart: dict, length: int):
    import statistics
    genes = loadAllRefGenes()
    scoreVec = []
    for gene in genes:
        for iso in gene.isoforms:
            codingSequence = iso.codingSeq
            for i in range(len(codingSequence) - length):
                subSeq = codingSequence[i:i+length]
                score = 0
                for j in range(len(chart)):
                    if subSeq in chart:
                        b = chart[subSeq]
                        score += b[1]
                scoreVec.append(score)
    return min(scoreVec), statistics.mean(scoreVec), max(scoreVec), statistics.stdev(scoreVec)



def loadGen(num: int):
    import pickle
    import os
    import datetime

    todaysDate = datetime.date.today()
    name = str(todaysDate) + 'gen' + str(num) + r'.pkl'
    root = r'C:\Users\Luke\PycharmProjects\GeneRider\GenAlgOut'
    fname = os.path.join(root, name)
    with open(fname, 'rb') as infile:
        gen = pickle.load(infile)
    infile.close()
    return gen

ModuleNotFoundError: No module named 'Standards'